In [7]:
import os
import numpy as np
import pickle
import pandas as pd
from scipy.stats import zscore
from brainbox.io.one import SessionLoader
from sklearn.preprocessing import StandardScaler
import gc
import concurrent.futures

from functions import get_left_licks, resample_common_time, interpolate_nans
from one.api import ONE
one = ONE(mode='remote')


In [8]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

# Get training sessions

In [9]:
# Get my functions
def idxs_from_files(design_matrices):
    
    idxs = []
    mouse_names = []
    for m, mat in enumerate(design_matrices):
        mouse_name = design_matrices[m][51:]
        eid = design_matrices[m][14:50]
        idx = str(eid + '_' + mouse_name)

        if len(idxs) == 0:
            idxs = idx
            mouse_names = mouse_name
        else:
            idxs = np.hstack((idxs, idx))
            mouse_names = np.hstack((mouse_names, mouse_name))
            
    return idxs, mouse_names

data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices'
all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

learning_data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/training_data/'
all_files = os.listdir(learning_data_path)
training_mice = []

for m, mouse_name in enumerate(np.unique(mouse_names)):
    
    filename = 'training_data_trials_'+mouse_name
    if filename in all_files:
        training_mice.append(mouse_name)
print(len(training_mice))

# List to store the filtered DataFrames for each mouse
all_filtered_sessions = []
all_filtered_mice = []
for mouse_name in training_mice:
    # 1. Load data
    file_path = f"{learning_data_path}training_data_trials_{mouse_name}"
    try:
        mouse_data = pd.read_parquet(file_path)
    except:
        print(f"Skipping {mouse_name}: File not found.")
        continue
    # 2. Ensure datetime and sort chronologically
    mouse_data['session_date'] = pd.to_datetime(mouse_data['session_date'])
    mouse_data = mouse_data.sort_values('session_date')
    # 3. Define the 3-day window
    start_date = mouse_data['session_date'].min()
    threshold = start_date + pd.Timedelta(days=3)
    # 4. Filter: Keep all sessions within the window
    # This keeps every trial/row that occurred in those 3 days
    mask = mouse_data['session_date'] <= threshold
    filtered_mouse_data = mouse_data.loc[mask].copy()
    mouse_sessions = filtered_mouse_data.session.unique()
    # mouse_sessions = mouse_data.session.unique()
    all_filtered_sessions.extend(mouse_sessions)
    all_filtered_mice.extend([mouse_name])

training_sessions = all_filtered_sessions

102
Skipping CSHL060: File not found.


In [40]:
sessions = training_sessions
data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices/training/'

os.chdir(data_path)
files = os.listdir()
sessions_to_process = []

for s, sess in enumerate(sessions):
    file_path = one.eid2path(sess)

    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    filename = "design_matrix_" + str(sess) + '_'  + mouse_name
    if filename not in files:
        sessions_to_process.append((sess))

len(sessions_to_process)

0

In [35]:
def process_design_matrix(session):

    file_path = one.eid2path(session)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    """ LOAD VARIABLES """
    sl = SessionLoader(eid=session, one=one)
    try:
        sl.load_session_data(trials=True, wheel=True, motion_energy=False)

        # Check if all data is available
        if np.sum(sl.data_info['is_loaded']) >= 1:

            # Wheel
            wheel = sl.wheel
            wheel_t = np.asarray(wheel['times'], dtype=np.float64)
            wheel_vel = wheel['velocity'].astype(np.float32)

            # Common resampling window
            onset = wheel_t.min()
            offset = wheel_t.max()
            fs = 30
            ref_t = np.arange(onset, offset, 1 / fs, dtype=np.float64)

            # Restrict to time window
            def restrict(t, x):
                mask = (t >= onset) & (t <= offset)
                return t[mask], x[mask]

            wheel_t, wheel_vel = restrict(wheel_t, wheel_vel)

            # Resample
            wh_down, rt = resample_common_time(ref_t, wheel_t, wheel_vel, kind='linear')

            # Create design matrix
            design_matrix = pd.DataFrame({
                'Bin': rt,
                'avg_wheel_vel': wh_down
            })

            # """ LOAD TRIAL DATA """
            session_trials = sl.trials
            session_start = list(session_trials['goCueTrigger_times'])[0]
            design_matrix = design_matrix.loc[(design_matrix['Bin'] > session_start)]

            """ SAVE DATA """       
            # Save unnormalized design matrix
            filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
            design_matrix.to_parquet(filename, compression='gzip')  

            # Save trials
            filename = data_path + "session_trials_" + str(session) + '_'  + mouse_name
            session_trials.to_parquet(filename, compression='gzip')  
            
            del design_matrix, session_trials, sl
            gc.collect()

        else:
            print('Data missing for session '+session)  
    except Exception as e:
        # We catch the exception, print the specific error, and move to the next session
        print(f"Failed to load session {session}. Error: {e}")


def parallel_process_data(sessions, function_name):
    with concurrent.futures.ThreadPoolExecutor() as executor:

        # Process each chunk in parallel
        executor.map(function_name, sessions)

In [ ]:
for s, session in enumerate(sessions_to_process):
    process_design_matrix(session)

2026-05-07 14:20:07 INFO     one.py:1288 Loading trials data


2026-05-07 14:20:08 INFO     one.py:1288 Loading wheel data
2026-05-07 14:20:09 INFO     one.py:1288 Loading pose data
2026-05-07 14:20:09 WARNING  one.py:1292 Could not load pose data.
2026-05-07 14:20:09 INFO     one.py:1288 Loading pupil data
2026-05-07 14:20:09 INFO     one.py:1450 Pupil diameter not available, trying to compute on the fly.
2026-05-07 14:20:09 WARNING  one.py:1292 Could not load pupil data.
